#### Simple GEN AI APP using LangChain

1) Load the data
    - loader.load()
2) Divide text into chunks
3) Convert Chunks into vector Embedding
4) Store the Vector Embeddings into Vectore store DB

In [ ]:
# Loading all the keys and setting up the environmemt variables

import os
from dotenv import load_dotenv

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# Used for LangSmith Tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')

In [2]:
## Data Ingestion -- From website scrape the data
from langchain_community.document_loaders import WebBaseLoader

/Users/mohinikathrotiya/Mohiniiiiiiiii/Python /Krish Naik- Udemy/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader = WebBaseLoader('https://docs.langchain.com/langsmith/observability-quickstart')
loader

In [4]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-quickstart', 'title': 'Tracing quickstart - Docs by LangChain', 'language': 'en'}, page_content='Tracing quickstart - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTracing quickstartGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewQuickstartConceptsTrace a RAG applicationTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesQuery traces (SDK)Compare tracesShare or unshare a trace publiclyRetrieve traces via CLIView server logs for a traceBulk export trace dataAutomationsSet up automation rulesConfigure webhook notifications for rulesFeedback & evaluationLog user feedback using the SDKSet up online

In [5]:
docs[0].page_content

'Tracing quickstart - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTracing quickstartGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewQuickstartConceptsTrace a RAG applicationTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesQuery traces (SDK)Compare tracesShare or unshare a trace publiclyRetrieve traces via CLIView server logs for a traceBulk export trace dataAutomationsSet up automation rulesConfigure webhook notifications for rulesFeedback & evaluationLog user feedback using the SDKSet up online evaluatorsMonitoring & alertingMonitor projects with dashboardsAlertsConfigure webhook notifications for alertsInsightsData type referenceRun (span) data formatFeedback data for

In [ ]:
# Making Text Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, chunk_overlap = 200
)
documents = splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-quickstart', 'title': 'Tracing quickstart - Docs by LangChain', 'language': 'en'}, page_content='Tracing quickstart - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTracing quickstartGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewQuickstartConceptsTrace a RAG applicationTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesQuery traces (SDK)Compare tracesShare or unshare a trace publiclyRetrieve traces via CLIView server logs for a traceBulk export trace dataAutomationsSet up automation rulesConfigure webhook notifications for rulesFeedback & evaluationLog user feedback using the SDKSet up online

In [7]:
## Converting Text Chunks into Vectors

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [8]:
## Storing Vectors into FAISS Vector DB
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(documents, embeddings)

In [9]:
vector_db

In [33]:
vector_db.index.d

1536

In [ ]:
# Query from a VectorDB

query = 'LangSmith addresses this by providing end-to-end visibility into how your application handles a request'
result = vector_db.similarity_search(query)
result[0].page_content

'LangSmith addresses this by providing end-to-end visibility into how your application handles a request. Each request generates a trace, which captures the full record of what happened. Within a trace are individual runs, the specific operations your application performed, such as an LLM call or a retrieval step. Tracing runs allows you to inspect, debug, and validate your application’s behavior.\nIn this quickstart, you will set up a minimal Retrieval Augmented Generation (RAG) application and add tracing with LangSmith. You will:'

In [34]:
result

[Document(id='075024dd-c073-4065-8c24-f1bed293a9ea', metadata={'source': 'https://docs.langchain.com/langsmith/observability-quickstart', 'title': 'Tracing quickstart - Docs by LangChain', 'language': 'en'}, page_content='LangSmith addresses this by providing end-to-end visibility into how your application handles a request. Each request generates a trace, which captures the full record of what happened. Within a trace are individual runs, the specific operations your application performed, such as an LLM call or a retrieval step. Tracing runs allows you to inspect, debug, and validate your application’s behavior.\nIn this quickstart, you will set up a minimal Retrieval Augmented Generation (RAG) application and add tracing with LangSmith. You will:'),
 Document(id='cefafb6a-5b31-4f8a-ad2c-1fb82a7133d2', metadata={'source': 'https://docs.langchain.com/langsmith/observability-quickstart', 'title': 'Tracing quickstart - Docs by LangChain', 'language': 'en'}, page_content='Tracing quickst

In [18]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = 'gpt-4o')

In [19]:
# Retrieval Chain, Document Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>
"""
)

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.comple

In [ ]:
# Manual Test of Document Chain

from langchain_core.documents import Document
document_chain.invoke({
    'input': 'LangSmith addresses this by providing end-to-end visibility into how your application handles a request',
    'context' : [Document(page_content= 'LangSmith addresses this by providing end-to-end visibility into how your application handles a request. Each request generates a trace, which captures the full record of what happened.')]
})

'LangSmith provides end-to-end visibility into how an application handles a request by generating a trace for each request, which captures a complete record of the events that occurred.'

In [ ]:
## Retriever: Converting FAISS into a retriever
## Job of retiever is to take a query and return relevant documents

retriever = vector_db.as_retriever()

In [ ]:
# Created Retrieval Chain

from langchain_classic.chains import create_retrieval_chain

retriever_chain = create_retrieval_chain(retriever, document_chain)
retriever_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1664178c0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
            | ChatOpenAI(

In [27]:
## Get teh response from the LLM 

response = retriever_chain.invoke({'input': 'LangSmith addresses this by providing end-to-end visibility into how your application handles a request'})
response['answer']

"What is the purpose of LangSmith according to the provided context?\n\nLangSmith provides end-to-end visibility into how your application handles requests by generating traces. Each trace captures the full record of what happened during a request, which includes individual runs or specific operations like LLM calls or retrieval steps. This tracing allows for inspection, debugging, and validation of the application's behavior, making it especially useful for applications built with Large Language Models (LLMs)."

In [28]:
response

{'input': 'LangSmith addresses this by providing end-to-end visibility into how your application handles a request',
 'context': [Document(id='075024dd-c073-4065-8c24-f1bed293a9ea', metadata={'source': 'https://docs.langchain.com/langsmith/observability-quickstart', 'title': 'Tracing quickstart - Docs by LangChain', 'language': 'en'}, page_content='LangSmith addresses this by providing end-to-end visibility into how your application handles a request. Each request generates a trace, which captures the full record of what happened. Within a trace are individual runs, the specific operations your application performed, such as an LLM call or a retrieval step. Tracing runs allows you to inspect, debug, and validate your application’s behavior.\nIn this quickstart, you will set up a minimal Retrieval Augmented Generation (RAG) application and add tracing with LangSmith. You will:'),
  Document(id='cefafb6a-5b31-4f8a-ad2c-1fb82a7133d2', metadata={'source': 'https://docs.langchain.com/langsm

In [29]:
response['context']

[Document(id='075024dd-c073-4065-8c24-f1bed293a9ea', metadata={'source': 'https://docs.langchain.com/langsmith/observability-quickstart', 'title': 'Tracing quickstart - Docs by LangChain', 'language': 'en'}, page_content='LangSmith addresses this by providing end-to-end visibility into how your application handles a request. Each request generates a trace, which captures the full record of what happened. Within a trace are individual runs, the specific operations your application performed, such as an LLM call or a retrieval step. Tracing runs allows you to inspect, debug, and validate your application’s behavior.\nIn this quickstart, you will set up a minimal Retrieval Augmented Generation (RAG) application and add tracing with LangSmith. You will:'),
 Document(id='cefafb6a-5b31-4f8a-ad2c-1fb82a7133d2', metadata={'source': 'https://docs.langchain.com/langsmith/observability-quickstart', 'title': 'Tracing quickstart - Docs by LangChain', 'language': 'en'}, page_content='Tracing quickst